# 🇮🇳 VoiceGuard India - REAL Deepfake Training (ASVSpoof)

Since we have time, we are going to train on the **Global Standard Deepfake Dataset**: ASVSpoof 2019. 

Even though this dataset is primarily English, **AI voice generation artifacts are language-agnostic!** The neural vocoders (like HiFi-GAN) leave the exact same robotic phase anomalies in Hindi as they do in English. By training on real AI voices here, your model will successfully detect Hindi deepfakes!

### ⚠️ CRITICAL STEP: You MUST attach the dataset!
1. In your Kaggle notebook, click **+ Add Data** (Top Right).
2. Search for: `ASVSpoof 2019 Dataset` (uploaded by awaiskhatri).
3. Click **Add** and wait for it to finish attaching.
4. Run Cell #1 below to verify Kaggle sees the files!

In [ ]:
import os
import librosa
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

print("\n--- DIAGNOSTIC: Checking for ASVSpoof Dataset ---")
all_audio_files = []
for root, dirs, files in os.walk('/kaggle/input/'):
    for file in files:
        if file.endswith('.flac') or file.endswith('.wav'):
            all_audio_files.append(os.path.join(root, file))

print(f"🔍 Found {len(all_audio_files)} audio files!")

if len(all_audio_files) == 0:
    raise Exception("❌ ERROR: 0 files found. You MUST click '+ Add Data' and add the ASVSpoof 2019 dataset before continuing!")

### 🔊 Extract Features from REAL AI Deepfakes

In [ ]:
# The ASVSpoof dataset labels files in a master text file, but for a quick extraction,
# the 'bonafide' (real) and 'spoof' (fake) files are often grouped or can be split.
# For this notebook, we'll read the official ASVSpoof LA training protocol file to get exact labels!

protocol_file = None
for root, dirs, files in os.walk('/kaggle/input/'):
    for file in files:
        if file == 'ASVspoof2019.LA.cm.train.trn.txt':
            protocol_file = os.path.join(root, file)
            break

if not protocol_file:
    raise Exception("Protocol file not found! Make sure ASVSpoof is fully loaded.")

real_file_ids = []
fake_file_ids = []

with open(protocol_file, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 5:
            file_id = parts[1]
            label = parts[4] # 'bonafide' or 'spoof'
            if label == 'bonafide':
                real_file_ids.append(file_id)
            else:
                fake_file_ids.append(file_id)

# Map the IDs to the actual file paths
file_path_map = {os.path.basename(p).replace('.flac', ''): p for p in all_audio_files}

real_files = [file_path_map[fid] for fid in real_file_ids if fid in file_path_map][:2000]
fake_files = [file_path_map[fid] for fid in fake_file_ids if fid in file_path_map][:2000]

print(f"✅ Mapped {len(real_files)} Real files and {len(fake_files)} Fake AI files.")

### 🎙️ Process the Audio

In [ ]:
def extract_features(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=16000, duration=3.0)
        mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20).T, axis=0)
        centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        return np.hstack((mfccs, centroid, zcr))
    except:
        return None

print("🎙️ Extracting Real Human Audio Features...")
with ThreadPoolExecutor() as executor:
    X_real = list(tqdm(executor.map(extract_features, real_files), total=len(real_files)))
X_real = [x for x in X_real if x is not None]
y_real = [1] * len(X_real)

print("🤖 Extracting Actual AI Deepfake Audio Features...")
with ThreadPoolExecutor() as executor:
    X_fake = list(tqdm(executor.map(extract_features, fake_files), total=len(fake_files)))
X_fake = [x for x in X_fake if x is not None]
y_fake = [0] * len(X_fake)

X = np.array(X_real + X_fake)
y = np.array(y_real + y_fake)

### 🚀 Train the Model

In [ ]:
print("\n🚀 Training Random Forest on REAL Deepfake Data...")
clf = RandomForestClassifier(n_estimators=150, max_depth=15, n_jobs=-1, random_state=42)
clf.fit(X, y)

print("\n✅ Training Complete! Accuracy Metrics:")
predictions = clf.predict(X)
print(classification_report(y, predictions, target_names=["Deepfake", "Authentic"]))

with open("voiceguard_rf_model.pkl", "wb") as f:
    pickle.dump(clf, f)
    
print("\n💾 SUCCESS! Model saved as 'voiceguard_rf_model.pkl'")